# Asymmetric Loss for conservative prediction

** Purpuse: minimize the underestimation risk - grid operators prefer over-provisioning**

* Option 1 — Post-processing (no retraining) - Pros: instant, no retraining. Cons: static offset, doesn't adapt to context.
* Option 2 — Quantile regression (retrain)

In [ ]:
# Option 1 — Post-processing (no retraining)

# On a validation set, collect residuals: residual = actual - predicted
# If residuals skew positive (you under-predict), add a percentile offset
residuals = y_val - model.predict(X_val)
offset = np.percentile(residuals, 75)   # moves ~75% of predictions above actual
preds_conservative = model.predict(X) + offset

In [ ]:
# Option 2 — Quantile regression (retrain)

# LightGBM — predict 70th percentile instead of mean
import lightgbm as lgb
model_conservative = lgb.LGBMRegressor(
    objective='quantile',
    alpha=0.70,        # tune: 0.6–0.8
    **other_params
)

# XGBoost
import xgboost as xgb
model_conservative = xgb.XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=0.70,
    **other_params
)

# Random Forest — predict upper bound of confidence interval
from sklearn.quantile-forest import  RandomForestQuantileRegressor
model_conservative = RandomForestQuantileRegressor(
    quantile=0.70,
    **other_params
)

